# Combinational Adders — Drawing the Circuit, Not Just the Table

This notebook builds arithmetic combinational circuits and **draws them as gate-level schematics** with the active wires lit up for the current inputs. Boolean correctness is checked in code; the focus is on *seeing the gates and the carry flow*.

$$S = A \oplus B \oplus C_{in}, \qquad C_{out} = AB + C_{in}(A \oplus B)$$


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.spines.bottom': False,
    'font.size': 9,
})

ON, OFF = '#c0392b', '#b0b0b0'
def wcol(bit): return ON if bit else OFF       # wire colour by logic level
def wlw(bit):  return 2.4 if bit else 1.2      # wire width by logic level


## Gate Glyphs

Each gate is drawn as a labelled box on an axis; wires are coloured by their current logic level — **red = 1, grey = 0**. A small helper draws a gate box and returns its input/output anchor points so wires can be routed between gates.


In [2]:
def gate_box(ax, x, y, label, w=0.9, h=0.7, out_bit=0):
    """Draw a gate as a rounded box; return (in_top, in_bot, out) anchor points."""
    rect = Rectangle((x, y - h/2), w, h, fc='#eef2f7', ec='#34495e', lw=1.4, zorder=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y, label, ha='center', va='center', fontsize=8.5,
            weight='bold', zorder=3)
    return (x, y + h*0.28), (x, y - h*0.28), (x + w, y)

def wire(ax, p0, p1, bit, elbow=True):
    """Route an orthogonal wire from p0 to p1, coloured by bit."""
    c, lw = wcol(bit), wlw(bit)
    (x0, y0), (x1, y1) = p0, p1
    if elbow and abs(y0 - y1) > 1e-6:
        xm = (x0 + x1) / 2
        ax.plot([x0, xm], [y0, y0], color=c, lw=lw, zorder=1)
        ax.plot([xm, xm], [y0, y1], color=c, lw=lw, zorder=1)
        ax.plot([xm, x1], [y1, y1], color=c, lw=lw, zorder=1)
    else:
        ax.plot([x0, x1], [y0, y1], color=c, lw=lw, zorder=1)

def pin(ax, x, y, name, bit, side='left'):
    """Draw an input/output terminal with its current bit value."""
    ax.scatter([x], [y], s=42, color=wcol(bit), zorder=4)
    dx = -0.28 if side == 'left' else 0.18
    ha = 'right' if side == 'left' else 'left'
    ax.text(x + dx, y, f'{name}={bit}', ha=ha, va='center', fontsize=9,
            color=wcol(bit), weight='bold')

print('drawing helpers ready')


drawing helpers ready


## Half Adder — One XOR, One AND

The half adder sums two bits with no carry-in. Toggle $A$ and $B$ and watch which wires light up: the XOR produces the sum, the AND produces the carry.

$$S = A \oplus B, \qquad C = A \cdot B$$


In [3]:
def draw_half_adder(A, B):
    S = A ^ B
    C = A & B
    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    ax.set_xlim(0, 7); ax.set_ylim(0, 4); ax.axis('off')

    pin(ax, 0.6, 2.7, 'A', A); pin(ax, 0.6, 1.3, 'B', B)
    xi_t, xi_b, xo = gate_box(ax, 2.6, 2.7, 'XOR', out_bit=S)
    ai_t, ai_b, ao = gate_box(ax, 2.6, 1.3, 'AND', out_bit=C)

    wire(ax, (0.6, 2.7), xi_t, A); wire(ax, (0.6, 1.3), xi_b, B)
    wire(ax, (0.6, 2.7), ai_t, A); wire(ax, (0.6, 1.3), ai_b, B)
    wire(ax, xo, (5.6, 2.7), S, elbow=False)
    wire(ax, ao, (5.6, 1.3), C, elbow=False)
    pin(ax, 5.6, 2.7, 'S', S, side='right')
    pin(ax, 5.6, 1.3, 'C', C, side='right')
    ax.set_title(f'Half Adder   {A}+{B} = {C}{S} (binary)', fontsize=10)
    plt.tight_layout(); plt.show()

wha_A = widgets.ToggleButtons(options=[0,1], value=1, description='A:')
wha_B = widgets.ToggleButtons(options=[0,1], value=1, description='B:')
display(widgets.VBox([wha_A, wha_B]),
        widgets.interactive_output(draw_half_adder, {'A': wha_A, 'B': wha_B}))


Output()

## Full Adder — Two Half Adders + an OR

A full adder chains two half adders. The first XORs $A,B$; the second XORs that with $C_{in}$ to give the sum. Either AND can raise the carry, so they feed an OR. Toggle all three inputs and trace the internal nodes lighting up.

$$S = (A \oplus B) \oplus C_{in}, \qquad C_{out} = (A\cdot B) + \big((A\oplus B)\cdot C_{in}\big)$$


In [4]:
def draw_full_adder(A, B, Cin):
    p = A ^ B           # first XOR
    g = A & B           # first AND (generate)
    S = p ^ Cin
    pc = p & Cin        # second AND (propagate carry)
    Cout = g | pc
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    ax.set_xlim(0, 10); ax.set_ylim(0, 5); ax.axis('off')

    pin(ax, 0.5, 4.0, 'A', A); pin(ax, 0.5, 3.2, 'B', B); pin(ax, 0.5, 1.0, 'Cin', Cin)
    x1t, x1b, x1o = gate_box(ax, 2.2, 3.6, 'XOR', out_bit=p)   # A^B
    a1t, a1b, a1o = gate_box(ax, 2.2, 2.0, 'AND', out_bit=g)   # A&B
    x2t, x2b, x2o = gate_box(ax, 5.0, 3.0, 'XOR', out_bit=S)   # p^Cin
    a2t, a2b, a2o = gate_box(ax, 5.0, 1.4, 'AND', out_bit=pc)  # p&Cin
    o1t, o1b, o1o = gate_box(ax, 7.6, 1.7, 'OR', out_bit=Cout) # g|pc

    wire(ax, (0.5, 4.0), x1t, A); wire(ax, (0.5, 3.2), x1b, B)
    wire(ax, (0.5, 4.0), a1t, A); wire(ax, (0.5, 3.2), a1b, B)
    wire(ax, x1o, x2t, p); wire(ax, (0.5, 1.0), x2b, Cin)
    wire(ax, x1o, a2t, p); wire(ax, (0.5, 1.0), a2b, Cin)
    wire(ax, a1o, o1t, g); wire(ax, a2o, o1b, pc)
    wire(ax, x2o, (9.2, 3.0), S, elbow=False)
    wire(ax, o1o, (9.2, 1.7), Cout, elbow=False)
    pin(ax, 9.2, 3.0, 'S', S, side='right')
    pin(ax, 9.2, 1.7, 'Cout', Cout, side='right')
    ax.set_title(f'Full Adder   {A}+{B}+{Cin} = {Cout}{S} (binary)', fontsize=10)
    plt.tight_layout(); plt.show()

wfa_A = widgets.ToggleButtons(options=[0,1], value=1, description='A:')
wfa_B = widgets.ToggleButtons(options=[0,1], value=1, description='B:')
wfa_C = widgets.ToggleButtons(options=[0,1], value=0, description='Cin:')
display(widgets.VBox([wfa_A, wfa_B, wfa_C]),
        widgets.interactive_output(draw_full_adder, {'A': wfa_A, 'B': wfa_B, 'Cin': wfa_C}))


Output()

## Ripple-Carry Adder — Watch the Carry Propagate

$N$ full adders chained carry-out to carry-in. Each stage is drawn as a block; the **carry chain along the bottom is the critical path**. Set two $N$-bit numbers and see exactly which carries are active and how far the chain reaches.

$$C_0 = C_{in}, \qquad C_{i+1} = A_iB_i + C_i(A_i \oplus B_i)$$


In [5]:
def draw_ripple(a_val, b_val, nbits):
    A = [(a_val >> i) & 1 for i in range(nbits)]
    B = [(b_val >> i) & 1 for i in range(nbits)]
    carries = [0]; S = []
    for i in range(nbits):
        s = A[i] ^ B[i] ^ carries[i]
        c = (A[i] & B[i]) | (carries[i] & (A[i] ^ B[i]))
        S.append(s); carries.append(c)
    total = a_val + b_val

    fig, ax = plt.subplots(figsize=(1.7 * nbits + 1.5, 3.6))
    ax.set_xlim(-0.5, 1.7 * nbits + 1); ax.set_ylim(0, 4); ax.axis('off')
    for i in range(nbits):                      # MSB on the left
        col = nbits - 1 - i
        x = col * 1.7
        rect = Rectangle((x, 1.4), 1.3, 1.2, fc='#eef2f7', ec='#34495e', lw=1.4, zorder=2)
        ax.add_patch(rect)
        ax.text(x + 0.65, 2.0, f'FA{i}', ha='center', va='center', weight='bold', fontsize=9)
        # A, B inputs from top
        ax.text(x + 0.3, 3.1, f'A{A[i]}', ha='center', color=wcol(A[i]), fontsize=8, weight='bold')
        ax.text(x + 1.0, 3.1, f'B{B[i]}', ha='center', color=wcol(B[i]), fontsize=8, weight='bold')
        ax.plot([x+0.3, x+0.3], [2.6, 3.0], color=wcol(A[i]), lw=wlw(A[i]))
        ax.plot([x+1.0, x+1.0], [2.6, 3.0], color=wcol(B[i]), lw=wlw(B[i]))
        # sum output downward
        ax.plot([x+0.65, x+0.65], [1.4, 0.8], color=wcol(S[i]), lw=wlw(S[i]))
        ax.text(x+0.65, 0.55, f'S{S[i]}', ha='center', color=wcol(S[i]), fontsize=9, weight='bold')
        # carry wire into this stage (from the right neighbour)
        cin = carries[i]
        ax.annotate('', xy=(x+1.3, 2.0), xytext=(x+1.7, 2.0),
                    arrowprops=dict(arrowstyle='->', color=wcol(cin), lw=wlw(cin)))
        ax.text(x+1.5, 2.22, str(cin), ha='center', color=wcol(cin), fontsize=8, weight='bold')
    # final carry-out on the far left
    cout = carries[nbits]
    ax.annotate('', xy=(-0.5, 2.0), xytext=(0.0, 2.0),
                arrowprops=dict(arrowstyle='->', color=wcol(cout), lw=wlw(cout)))
    ax.text(-0.35, 2.25, f'Cout={cout}', ha='center', color=wcol(cout), fontsize=8, weight='bold')
    ax.set_title(f'{a_val} + {b_val} = {total}   (carry chain depth = critical path)', fontsize=10)
    plt.tight_layout(); plt.show()

wn = widgets.IntSlider(value=4, min=2, max=6, description='bits N:')
def _mk(nbits):
    hi = (1 << nbits) - 1
    wa = widgets.IntSlider(value=min(11, hi), min=0, max=hi, description='A:')
    wb = widgets.IntSlider(value=min(7, hi), min=0, max=hi, description='B:')
    out = widgets.interactive_output(draw_ripple, {'a_val': wa, 'b_val': wb, 'nbits': widgets.fixed(nbits)})
    display(widgets.VBox([wa, wb]), out)
display(wn, widgets.interactive_output(_mk, {'nbits': wn}))


IntSlider(value=4, description='bits N:', max=6, min=2)

Output()

## Ripple vs Carry-Look-Ahead — Why the Schematic Shape Matters

The ripple adder's carry must pass through every stage in series, so worst-case delay grows linearly with $N$. A carry-look-ahead unit computes all carries in parallel from generate/propagate terms, trading more gates for near-constant carry depth. Same arithmetic, very different critical path.

$$g_i = A_iB_i,\quad p_i = A_i \oplus B_i,\qquad C_{i+1} = g_i + p_i C_i$$


In [6]:
def compare_delay(nmax):
    N = np.arange(1, nmax + 1)
    ripple = 2 * N + 1          # ~2 gate delays per stage carry path
    cla = 2 + 2 * np.ceil(np.log2(N + 1))  # log-depth carry tree (schematic)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    ax.plot(N, ripple, 'o-', color='#c0392b', lw=2, label='ripple-carry (series)')
    ax.plot(N, cla, 's-', color='#2471a3', lw=2, label='carry-look-ahead (parallel)')
    ax.set_xlabel('adder width N (bits)'); ax.set_ylabel('worst-case gate delays')
    ax.grid(True, alpha=0.3); ax.legend()
    ax.set_title('Critical-path delay vs adder width')
    plt.tight_layout(); plt.show()

wnmax = widgets.IntSlider(value=16, min=4, max=64, step=4, description='max N:')
display(wnmax, widgets.interactive_output(compare_delay, {'nmax': wnmax}))


IntSlider(value=16, description='max N:', max=64, min=4, step=4)

Output()

## 1-Bit Magnitude Comparator

A comparator outputs three mutually exclusive flags. For a single bit: $A>B$ needs $A\overline{B}$, $A<B$ needs $\overline{A}B$, and equality is the XNOR. The active flag's wire lights up.

$$ (A>B)=A\overline{B}, \quad (A<B)=\overline{A}B, \quad (A=B)=\overline{A\oplus B}$$


In [7]:
def draw_comparator(A, B):
    gt = A & (1 - B)
    lt = (1 - A) & B
    eq = 1 - (A ^ B)
    fig, ax = plt.subplots(figsize=(7, 3.8))
    ax.set_xlim(0, 8); ax.set_ylim(0, 5); ax.axis('off')
    pin(ax, 0.6, 3.5, 'A', A); pin(ax, 0.6, 1.5, 'B', B)
    labels = [('A>B', gt, 4.0), ('A=B', eq, 2.5), ('A<B', lt, 1.0)]
    for name, bit, y in labels:
        _, _, o = gate_box(ax, 3.2, y, name.replace('A','').replace('B',''), w=1.0, out_bit=bit)
        wire(ax, (0.6, 3.5), (3.2, y + 0.18), A)
        wire(ax, (0.6, 1.5), (3.2, y - 0.18), B)
        wire(ax, o, (6.0, y), bit, elbow=False)
        pin(ax, 6.0, y, name, bit, side='right')
    rel = '>' if gt else ('<' if lt else '=')
    ax.set_title(f'Comparator:  A {rel} B   (A={A}, B={B})', fontsize=10)
    plt.tight_layout(); plt.show()

wc_A = widgets.ToggleButtons(options=[0,1], value=1, description='A:')
wc_B = widgets.ToggleButtons(options=[0,1], value=0, description='B:')
display(widgets.VBox([wc_A, wc_B]),
        widgets.interactive_output(draw_comparator, {'A': wc_A, 'B': wc_B}))


Output()

## Parity Generator — A Tree of XORs

Even parity appends the bit that makes the total number of 1s even; it is simply the XOR of all data bits. A balanced XOR tree gives logarithmic depth. The cell verifies the parity bit and the resulting word count for every input.

$$P_{even} = b_0 \oplus b_1 \oplus \dots \oplus b_{n-1}$$


In [8]:
def parity_tree(word, nbits):
    bits = [(word >> i) & 1 for i in range(nbits)]
    p = 0
    for b in bits: p ^= b
    full = bits + [p]
    ones = sum(full)
    fig, ax = plt.subplots(figsize=(7, 2.4))
    ax.set_xlim(-0.5, nbits + 1); ax.set_ylim(0, 2); ax.axis('off')
    for i, b in enumerate(bits):
        ax.scatter([i], [1.4], s=120, color=wcol(b), zorder=3)
        ax.text(i, 1.7, f'b{i}={b}', ha='center', fontsize=8, color=wcol(b), weight='bold')
    ax.scatter([nbits], [1.4], s=160, color=wcol(p), marker='D', zorder=3)
    ax.text(nbits, 1.75, f'P={p}', ha='center', fontsize=9, color=wcol(p), weight='bold')
    ax.text((nbits)/2, 0.5,
            f'total ones incl. parity = {ones}  -> {"EVEN OK" if ones % 2 == 0 else "ODD!"}',
            ha='center', fontsize=10,
            color='#2471a3' if ones % 2 == 0 else '#c0392b', weight='bold')
    plt.tight_layout(); plt.show()

# verify even parity makes total ones even for all 4-bit words
ok = all((bin(w).count('1') + (bin(w).count('1') & 1)) % 2 == 0 for w in range(16))
print(f'even-parity invariant holds for all 4-bit words: {ok}')

wp = widgets.IntSlider(value=0b1011, min=0, max=15, description='word:')
display(wp, widgets.interactive_output(parity_tree, {'word': wp, 'nbits': widgets.fixed(4)}))


even-parity invariant holds for all 4-bit words: True


IntSlider(value=11, description='word:', max=15)

Output()